<a href="https://colab.research.google.com/github/indobims/Pemrograman-Berorientasi-Object/blob/main/Jobsheet_11_Muhammad_Arya_Bima_Surya_Pratama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Nama : Muhammad Arya Bima Surya Pratama<br>
Kelas : Ti-1B<br>
Nim : 4.33.25.1.15

**Langkah 1** : Persiapan Awal (Konfigurasi & Setup Database)

In [ ]:
!pip install -q streamlit pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 97.4 MB/s eta 0:00:00


In [10]:
%%writefile konfigurasi.py
import os

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
NAMA_DB = 'pengeluaran_harian.db'
DB_PATH = os.path.join(BASE_DIR, NAMA_DB)
KATEGORI_PENGELUARAN = ["Makanan", "Transportasi", "Hiburan", "Tagihan", "Belanja", "Kesehatan", "Pendidikan", "Lainnya"]
KATEGORI_DEFAULT = "Lainnya"

Writing konfigurasi.py


In [1]:
%%writefile setup_db_pengeluaran.py
import sqlite3
import os
from konfigurasi import DB_PATH

def setup_database():
    print(f"Memeriksa/membuat database di: {DB_PATH}")
    conn = None
    try:
        conn = sqlite3.connect(DB_PATH)
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        deskripsi TEXT NOT NULL,
        jumlah REAL NOT NULL CHECK(jumlah > 0),
        kategori TEXT,
        tanggal DATE NOT NULL
        );"""
        print(" Membuat tabel 'transaksi' (jika belum ada)...")
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f" -> Error SQLite saat setup: {e}")
        return False
    finally:
        if conn:
            conn.close()
            print(" -> Koneksi DB setup ditutup.")

if __name__ == "__main__":
    print("--- Memulai Setup Database Pengeluaran ---")
    if setup_database():
        print(f"\nSetup database '{os.path.basename(DB_PATH)}' selesai.")
    else:
        print(f"\nSetup database GAGAL.")
    print("--- Setup Database Selesai ---")

Writing setup_db_pengeluaran.py


**Langkah 2** : Modul Akses Database (database.py)

In [2]:
%%writefile database.py
import sqlite3
import pandas as pd
from konfigurasi import DB_PATH

def get_db_connection() -> sqlite3.Connection | None:
    """Membuka dan mengembalikan koneksi baru ke database SQLite."""
    try:
        conn = sqlite3.connect(DB_PATH, timeout=10, detect_types=sqlite3.PARSE_DECLTYPES)
        conn.row_factory = sqlite3.Row
        return conn
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Koneksi DB gagal: {e}")
        return None

def execute_query(query: str, params: tuple = None):
    """Menjalankan query non-SELECT. Mengembalikan lastrowid jika INSERT."""
    conn = get_db_connection()
    if not conn: return None
    last_id = None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        conn.commit()
        last_id = cursor.lastrowid
        return last_id
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Query gagal: {e} | Query: {query[:60]}")
        conn.rollback()
        return None
    finally:
        if conn: conn.close()

def fetch_query(query: str, params: tuple = None, fetch_all: bool = True):
    """Menjalankan query SELECT dan mengembalikan hasil."""
    conn = get_db_connection()
    if not conn: return None
    try:
        cursor = conn.cursor()
        if params:
            cursor.execute(query, params)
        else:
            cursor.execute(query)
        result = cursor.fetchall() if fetch_all else cursor.fetchone()
        return result
    except sqlite3.Error as e:
        print(f"ERROR [database.py] Fetch gagal: {e} | Query: {query[:60]}")
        return None
    finally:
        if conn: conn.close()

def get_dataframe(query: str, params: tuple = None) -> pd.DataFrame:
    """Menjalankan query SELECT dan mengembalikan DataFrame Pandas."""
    conn = get_db_connection()
    if not conn: return pd.DataFrame()
    try:
        df = pd.read_sql_query(query, conn, params=params)
        return df
    except Exception as e:
        print(f"ERROR [database.py] Gagal baca ke DataFrame: {e}")
        return pd.DataFrame()
    finally:
        if conn: conn.close()

def setup_database_initial():
    """Memastikan tabel transaksi ada (dipanggil oleh AnggaranHarian jika perlu)."""
    print(f"Memeriksa/membuat tabel di database (via database.py): {DB_PATH}")
    conn = get_db_connection()
    if not conn: return False
    try:
        cursor = conn.cursor()
        sql_create_table = """
        CREATE TABLE IF NOT EXISTS transaksi (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        deskripsi TEXT NOT NULL,
        jumlah REAL NOT NULL CHECK(jumlah > 0),
        kategori TEXT,
        tanggal DATE NOT NULL );"""
        cursor.execute(sql_create_table)
        conn.commit()
        print(" -> Tabel 'transaksi' siap.")
        return True
    except sqlite3.Error as e:
        print(f"Error SQLite saat setup tabel: {e}")
        return False
    finally:
        if conn: conn.close()

Writing database.py


**Langkah 3** : Modul Model Data (model.py)

In [3]:
%%writefile model.py
import datetime

class Transaksi:
    """Merepresentasikan satu entitas transaksi pengeluaran (Data Class)."""
    def __init__(self, deskripsi: str, jumlah: float, kategori: str, tanggal: datetime.date | str, id_transaksi: int | None = None):
        self.id = id_transaksi
        self.deskripsi = str(deskripsi) if deskripsi else "Tanpa Deskripsi"
        try:
            jumlah_float = float(jumlah)
            self.jumlah = jumlah_float if jumlah_float > 0 else 0.0
            if jumlah_float <= 0:
                print(f"Peringatan: Jumlah '{jumlah}' harus positif.")
        except (ValueError, TypeError):
            self.jumlah = 0.0
            print(f"Peringatan: Jumlah '{jumlah}' tidak valid.")

        self.kategori = str(kategori) if kategori else "Lainnya"
        if isinstance(tanggal, datetime.date):
            self.tanggal = tanggal
        elif isinstance(tanggal, str):
            try:
                self.tanggal = datetime.datetime.strptime(tanggal, "%Y-%m-%d").date()
            except ValueError:
                self.tanggal = datetime.date.today()
                print(f"Peringatan: Format tgl '{tanggal}' salah.")
        else:
            self.tanggal = datetime.date.today()
            print(f"Peringatan: Tipe tgl '{type(tanggal)}' tidak valid.")

    def __repr__(self) -> str:
        try:
            import locale
            locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
            jml_str = locale.format_string("%.0f", self.jumlah, grouping=True)
        except:
            jml_str = f"{self.jumlah:.0f}"
        return f"Transaksi(ID:{self.id}, Tgl:{self.tanggal.strftime('%Y-%m-%d')}, Jml:{jml_str}, Kat:'{self.kategori}', Desc:'{self.deskripsi}')"

    def to_dict(self) -> dict:
        return {
            "deskripsi": self.deskripsi,
            "jumlah": self.jumlah,
            "kategori": self.kategori,
            "tanggal": self.tanggal.strftime("%Y-%m-%d")
        }


Writing model.py


**Langkah 4** : Modul Manajer Anggaran (manajer_anggaran.py)

In [4]:
%%writefile manajer_anggaran.py
import datetime
import pandas as pd
from model import Transaksi
import database

class AnggaranHarian:
    """Mengelola logika bisnis pengeluaran harian (Repository Pattern)."""
    _db_setup_done = False

    def __init__(self):
        if not AnggaranHarian._db_setup_done:
            if database.setup_database_initial():
                AnggaranHarian._db_setup_done = True

    def tambah_transaksi(self, transaksi: Transaksi) -> bool:
        if not isinstance(transaksi, Transaksi) or transaksi.jumlah <= 0:
            return False
        sql = "INSERT INTO transaksi (deskripsi, jumlah, kategori, tanggal) VALUES (?, ?, ?, ?)"
        params = (transaksi.deskripsi, transaksi.jumlah, transaksi.kategori, transaksi.tanggal.strftime("%Y-%m-%d"))
        last_id = database.execute_query(sql, params)
        if last_id is not None:
            transaksi.id = last_id
            return True
        return False

    def get_semua_transaksi_obj(self) -> list[Transaksi]:
        sql = "SELECT id, deskripsi, jumlah, kategori, tanggal FROM transaksi ORDER BY tanggal DESC, id DESC"
        rows = database.fetch_query(sql, fetch_all=True)
        transaksi_list = []
        if rows:
            for row in rows:
                transaksi_list.append(Transaksi(
                    id_transaksi=row['id'],
                    deskripsi=row['deskripsi'],
                    jumlah=row['jumlah'],
                    kategori=row['kategori'],
                    tanggal=row['tanggal']
                ))
        return transaksi_list

    def get_dataframe_transaksi(self, filter_tanggal: datetime.date | None = None) -> pd.DataFrame:
        query = "SELECT id, tanggal, kategori, deskripsi, jumlah FROM transaksi"
        params = None
        if filter_tanggal:
            query += " WHERE tanggal = ?"
            params = (filter_tanggal.strftime("%Y-%m-%d"),)
        query += " ORDER BY tanggal DESC, id DESC"

        df = database.get_dataframe(query, params=params)

        if not df.empty:
            try:
                import locale
                locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: locale.currency(x or 0, grouping=True, symbol='Rp ')[:-3])
            except:
                df['Jumlah (Rp)'] = df['jumlah'].map(lambda x: f"Rp {x or 0:,.0f}".replace(",", "."))
            # Pastikan kolom id ikut disertakan agar user bisa melihat ID mana yang akan dihapus
            df = df[['id', 'tanggal', 'kategori', 'deskripsi', 'Jumlah (Rp)']]
        return df

    def hitung_total_pengeluaran(self, tanggal: datetime.date | None = None) -> float:
        sql = "SELECT SUM(jumlah) FROM transaksi"
        params = None
        if tanggal:
            sql += " WHERE tanggal = ?"
            params = (tanggal.strftime("%Y-%m-%d"),)
        result = database.fetch_query(sql, params=params, fetch_all=False)
        if result and result[0] is not None:
            return float(result[0])
        return 0.0

    def get_pengeluaran_per_kategori(self, tanggal: datetime.date | None = None) -> dict:
        hasil = {}
        sql = "SELECT kategori, SUM(jumlah) FROM transaksi"
        params = []
        if tanggal:
            sql += " WHERE tanggal = ?"
            params.append(tanggal.strftime("%Y-%m-%d"))
        sql += " GROUP BY kategori HAVING SUM(jumlah) > 0 ORDER BY SUM(jumlah) DESC"
        rows = database.fetch_query(sql, params=tuple(params) if params else None, fetch_all=True)
        if rows:
            for row in rows:
                kategori = row['kategori'] if row['kategori'] else "Lainnya"
                jumlah = float(row[1]) if row[1] is not None else 0.0
                hasil[kategori] = jumlah
        return hasil

    # --- IMPLEMENTASI SOAL JOBSHEET: FUNGSI HAPUS ---
    def hapus_transaksi(self, id_transaksi: int) -> bool:
        """Menghapus data transaksi dari database berdasarkan ID"""
        sql = "DELETE FROM transaksi WHERE id = ?"
        params = (id_transaksi,)
        hasil = database.execute_query(sql, params)
        return hasil is not None

Writing manajer_anggaran.py


**Langkah 5** : Aplikasi Utama Streamlit (main_app.py)

In [5]:
%%writefile main_app.py
import streamlit as st
import datetime
import pandas as pd
import locale

# Setup Locale untuk format Rupiah
try:
    locale.setlocale(locale.LC_ALL, 'id_ID.UTF-8')
except locale.Error:
    try:
        locale.setlocale(locale.LC_ALL, 'Indonesian_Indonesia.1252')
    except:
        pass

def format_rp(angka):
    try:
        return locale.currency(angka or 0, grouping=True, symbol='Rp ')[:-3]
    except:
        return f"Rp {angka or 0:,.0f}".replace(",",".")

try:
    from model import Transaksi
    from manajer_anggaran import AnggaranHarian
    from konfigurasi import KATEGORI_PENGELUARAN
except ImportError as e:
    st.error(f"Gagal mengimpor modul: {e}. Pastikan file .py lain ada di direktori yang sama.")
    st.stop()

# Konfigurasi Halaman Utama
st.set_page_config(page_title="Catatan Pengeluaran", page_icon="💸", layout="wide", initial_sidebar_state="expanded")

@st.cache_resource
def get_anggaran_manager():
    return AnggaranHarian()

anggaran = get_anggaran_manager()

# Inisialisasi session_state untuk konfirmasi hapus
if 'hapus_id_konfirmasi' not in st.session_state:
    st.session_state.hapus_id_konfirmasi = None

def halaman_input(anggaran: AnggaranHarian):
    st.header("📝 Tambah Pengeluaran Baru")
    st.markdown("Masukkan detail pengeluaran harianmu di bawah ini.")

    with st.form("form_transaksi_baru", clear_on_submit=True):
        col1, col2 = st.columns([2, 1])
        with col1:
            deskripsi = st.text_input("Deskripsi Pengeluaran*", placeholder="Contoh: Makan siang nasi padang")
        with col2:
            kategori = st.selectbox("Kategori*:", KATEGORI_PENGELUARAN, index=0)

        col3, col4 = st.columns([1, 1])
        with col3:
            jumlah = st.number_input("Total Jumlah (Rp)*:", min_value=0.01, step=1000.0, format="%.0f", value=None, placeholder="Contoh: 25000")
        with col4:
            tanggal = st.date_input("Tanggal Transaksi*:", value=datetime.date.today())

        st.markdown("<br>", unsafe_allow_html=True)
        submitted = st.form_submit_button("💾 Simpan Transaksi", use_container_width=True)

        if submitted:
            if not deskripsi or jumlah is None or jumlah <= 0:
                st.warning("⚠️ Data belum lengkap atau jumlah tidak valid!")
            else:
                with st.spinner("Menyimpan ke database..."):
                    tx = Transaksi(deskripsi, float(jumlah), kategori, tanggal)
                    if anggaran.tambah_transaksi(tx):
                        st.success("✅ Berhasil! Transaksi telah disimpan.")
                        st.cache_data.clear()
                        st.rerun()
                    else:
                        st.error("❌ Gagal menyimpan transaksi.")

def halaman_riwayat(anggaran: AnggaranHarian):
    st.header("📜 Riwayat Lengkap")

    col_title, col_btn = st.columns([4, 1])
    with col_btn:
        if st.button("🔄 Segarkan Data", use_container_width=True):
            st.session_state.hapus_id_konfirmasi = None
            st.cache_data.clear()
            st.rerun()

    with st.spinner("Memuat riwayat transaksi..."):
        df_transaksi = anggaran.get_dataframe_transaksi()

    if df_transaksi is None or df_transaksi.empty:
        st.info("💡 Belum ada transaksi yang dicatat. Silakan tambah data terlebih dahulu.")
    else:
        # Menampilkan tabel secara responsif dengan st.dataframe
        st.dataframe(df_transaksi, use_container_width=True, hide_index=True)

        st.markdown("---")

        # --- IMPLEMENTASI SOAL JOBSHEET: METODE INPUT ID & TOMBOL HAPUS ---
        st.subheader("🗑️ Hapus Transaksi")
        st.caption("Masukkan **ID** dari tabel di atas untuk menghapus transaksi yang tidak diinginkan.")

        col_input, col_hapus = st.columns([2, 1])
        with col_input:
            id_hapus = st.number_input("Pilih ID Transaksi:", min_value=1, step=1, key="input_id_hapus")
        with col_hapus:
            st.write("") # Spacing
            st.write("") # Spacing
            if st.button("🗑️ Hapus Transaksi Terpilih", use_container_width=True):
                st.session_state.hapus_id_konfirmasi = id_hapus

        # Logika Konfirmasi (Sesuai Jobsheet)
        if st.session_state.hapus_id_konfirmasi is not None:
            id_target = st.session_state.hapus_id_konfirmasi
            st.warning(f"⚠️ **PERHATIAN:** Apakah Anda yakin ingin menghapus transaksi dengan ID **{id_target}**?", icon="⚠️")

            c1, c2 = st.columns(2)
            with c1:
                if st.button("✅ Ya, Konfirmasi Hapus", type="primary", use_container_width=True):
                    with st.spinner("Menghapus..."):
                        if anggaran.hapus_transaksi(id_target):
                            st.success(f"🎉 Transaksi ID {id_target} berhasil dihapus!")
                            st.session_state.hapus_id_konfirmasi = None
                            st.cache_data.clear()
                            st.rerun()
                        else:
                            st.error(f"❌ Gagal menghapus! Pastikan ID {id_target} benar-benar ada.")
            with c2:
                if st.button("❌ Batal", use_container_width=True):
                    st.session_state.hapus_id_konfirmasi = None
                    st.rerun()

def halaman_ringkasan(anggaran: AnggaranHarian):
    st.header("📊 Ringkasan Keuangan")

    # Filter Kartu
    with st.container():
        col_filter, col_metric = st.columns([1, 1])
        with col_filter:
            st.markdown("**🔍 Filter Data**")
            pilihan_periode = st.selectbox("Tampilkan berdasarkan periode:", ["Semua Waktu", "Hari Ini", "Pilih Tanggal Tertentu"], key="filter_periode", on_change=lambda: st.cache_data.clear())

            tanggal_filter = None
            label_periode = "Semua Waktu"

            if pilihan_periode == "Hari Ini":
                tanggal_filter = datetime.date.today()
                label_periode = f"Hari Ini ({tanggal_filter.strftime('%d %b %Y')})"
            elif pilihan_periode == "Pilih Tanggal Tertentu":
                if 'tanggal_pilihan_state' not in st.session_state:
                    st.session_state.tanggal_pilihan_state = datetime.date.today()
                tanggal_filter = st.date_input("Pilih Tanggal:", value=st.session_state.tanggal_pilihan_state, key="tanggal_pilihan", on_change=lambda: setattr(st.session_state, 'tanggal_pilihan_state', st.session_state.tanggal_pilihan) or st.cache_data.clear())
                label_periode = f"Tanggal ({tanggal_filter.strftime('%d %b %Y')})"

        with col_metric:
            @st.cache_data(ttl=300)
            def hitung_total_cached(tgl_filter):
                return anggaran.hitung_total_pengeluaran(tanggal=tgl_filter)

            total_pengeluaran = hitung_total_cached(tanggal_filter)
            st.metric(label=f"💰 Total Pengeluaran ({label_periode})", value=format_rp(total_pengeluaran))

    st.markdown("---")
    st.subheader(f"📈 Distribusi per Kategori")

    @st.cache_data(ttl=300)
    def get_kategori_cached(tgl_filter):
        return anggaran.get_pengeluaran_per_kategori(tanggal=tgl_filter)

    with st.spinner(f"Memuat grafik kategori..."):
        dict_per_kategori = get_kategori_cached(tanggal_filter)

    if not dict_per_kategori:
        st.info("📉 Tidak ada pengeluaran pada periode ini.")
    else:
        try:
            data_kategori = [{"Kategori": kat, "Total": jml} for kat, jml in dict_per_kategori.items()]
            df_kategori = pd.DataFrame(data_kategori).sort_values(by="Total", ascending=False).reset_index(drop=True)
            df_kategori['Total (Rp)'] = df_kategori['Total'].apply(format_rp)

            col_tabel, col_grafik = st.columns([1, 1.5])
            with col_tabel:
                st.dataframe(df_kategori[['Kategori', 'Total (Rp)']], hide_index=True, use_container_width=True)
            with col_grafik:
                st.bar_chart(df_kategori.set_index('Kategori')['Total'], use_container_width=True)
        except Exception as e:
            st.error(f"Gagal menampilkan grafik: {e}")

def main():
    # Desain Sidebar Baru
    with st.sidebar:
        st.title("💸 DompetKu")
        st.caption("Aplikasi Pencatatan Pengeluaran")
        st.markdown("---")

        menu_pilihan = st.radio("📌 Navigasi Menu:", ["Tambah", "Riwayat", "Ringkasan"], key="menu_utama")

        st.markdown("---")

        # Info Author
        st.markdown("### 👨‍💻 Informasi Author")
        st.info("""
        **Muhammad Arya Bima Surya Pratama** NIM: 4.33.25.1.15
        D4 T. Rekayasa Komputer - Polines
        """)

    manajer_anggaran = get_anggaran_manager()

    # Routing Halaman
    if menu_pilihan == "Tambah":
        halaman_input(manajer_anggaran)
    elif menu_pilihan == "Riwayat":
        halaman_riwayat(manajer_anggaran)
    elif menu_pilihan == "Ringkasan":
        halaman_ringkasan(manajer_anggaran)

if __name__ == "__main__":
    main()

Writing main_app.py


**Langkah 6** : Menjalankan dan Menguji Aplikasi Modular

In [9]:
!python setup_db_pengeluaran.py


Traceback (most recent call last):
  File "/content/setup_db_pengeluaran.py", line 3, in <module>
    from konfigurasi import DB_PATH
ModuleNotFoundError: No module named 'konfigurasi'


In [7]:
!wget -q -O - ipv4.icanhazip.com

136.114.32.77


In [14]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

In [16]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 111.6 MB/s eta 0:00:00


In [17]:

!pkill -f cloudflared
!pkill -f localtunnel
!pkill -f streamlit
!sleep 2

!nohup streamlit run main_app.py &>/content/logs.txt &
!sleep 3

!./cloudflared tunnel --url http://localhost:8501

2026-06-04T12:29:09Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-04T12:29:09Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-04T12:29:13Z INF +--------------------------------------------------------------------------------------------+
2026-06-04T12:29:13Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-04T12:29:13Z INF |  https://simple-stripes-equity-transform.trycloudflare